# Final Project 
My main goal is to try out visual poetry. 
Blog post with process: https://joysblog.cargo.site/rwet-final


In [51]:
#whole code
import sentencex
import re
import random
import textwrap
from wordfreq import top_n_list
from collections import Counter
from itertools import combinations
import string 

# markov chain based on line_block
# didn't end up using markov chain
# from shoestrings import TextGenerator
# from shoestrings.tokenizers import SimpleWordTokenizer, SimpleCharacterTokenizer

import pronouncing as pr

import tracery
from tracery.modifiers import base_english

# categerizing the sub words
import spacy
nlp = spacy.load("en_core_web_sm")

#processing& cleaning  my text
text = open("underground.txt").read()
text = text.replace("\r\n", "\n").replace("\r", "\n")
text = text.replace("\n", " ")
text = text.replace("_", "")
text = text.replace('\u201c', '').replace('\u201d', '')   # smart quotes
text = text.replace('\u2018', "'").replace('\u2019', "'")
text = text.replace("—", ". ")                            # em-dash → period
text = re.sub(r'\s+', ' ', text)    # collapse whitespace

sentences = sentencex.segment("en", text) 
sentences= [re.sub(r"[^\w\s,'\-.]", '', item) for item in sentences]
sentences = [item.strip() for item in sentences if len(item)>0]
words = text.split()

words = [w.strip(string.punctuation) for w in words]
words = [w for w in words if w]   # drop empty strings
long_words = [w for w in words if len(w) > 12] #11-12 words is the best?
#long_words = Counter(long_words).most_common(200)

#choosing the main word (the top word)
word_choice = random.choice(long_words)#|[0]
n = len(word_choice)



word = word_choice
print("chosen word: " + word)
print()

wordlist = top_n_list("en", 40000)
#random.choice(wordlist)

#checking all the possible subwords
results = []
seen_words = []

for i in range(n-1, 2,-1): #keep 2-(n-1) words
    positions = []
    #print(list(combinations(range(n),i)))
    for keep_positions in combinations(range(n),i):
        candidate = "".join(word[p] for p in keep_positions)
        #print(candidate)
        if candidate in wordlist and candidate not in seen_words:
            seen_words.append(candidate)
            results.append({"word":candidate, "position": keep_positions, "length": len(candidate)})
results.insert(0, {"word": word, "position": range(n), "length":len(word)})

found_lines = [] # store the line
used_subwords = [] #tracking sub-words that were used in the text
#max_words = 300
matching_lines = []
seen_sentences = []
used_subwords_word = [word_choice] # only the words

for item in results:
    sub = item["word"]

    matching  =[]
    for s in sentences:
        sentence_words = s.split() # word inside the sentences
        cleaned_words = [w.strip(string.punctuation).lower() for w in sentence_words]
        if sub in cleaned_words:    
        #if sub in cleaned_words and len(sentence_words) <= max_words:
            matching.append(s)
            
    # Add only new sentences to matching_lines
    for s in matching:
        if s not in seen_sentences:
            matching_lines.append(s)
            seen_sentences.append(s)
    
    if matching:
        used_subwords.append(item) 
    
#print(used_subwords)
#print(seen_sentences)

stars = ['✦', '✶', '✱', '✳', '✴', '*', '⊹', '·']
ASCII = ['*', '+', '.', ':', ';', '·', 'o', 'x', '^', '~']
# printing words and star version
for item in used_subwords:
    used_subwords_word.append(item["word"])
    keep_set = item["position"]
    #print(keep_set)
    line = ""
    for i in range(n): 
        if i in keep_set:
            line += " "
            line +=word[i]
        else:
            line += "  "
    print(line)

print()
print("----------- STAR -----------")
print()

for item in used_subwords:
    keep_set = item["position"]
    #print(keep_set)
    line = ""
    for i in range(n): 
        if i in keep_set:
            line += " "
            line += random.choice(ASCII)
        else:
            line += "  "
    print(line)

print()
print("----------- poem -----------")
print()

# categerizing the sub words
nlp = spacy.load("en_core_web_sm")
subwords_by_pos = {
    "NOUN": [],
    "VERB": [],
    "ADJ": [],
    "ADV": [],
    "ADP": [],   # preposition
    "INTJ": [], # interjection -> but not a lot found
    "PROPN": [] # not a lot found
}

#Spacy taggoing
for line in matching_lines:
    doc = nlp(line)
    for token in doc:
        word = token.text.lower()
        #print(word)
        if word in used_subwords_word:
            pos = token.pos_ 
            #pos = token.tag_ <- the other kind of tagging strataey? didn't notice too much of differrent
            if pos in subwords_by_pos and word not in subwords_by_pos[pos]:
                subwords_by_pos[pos].append(word)

for pos, words in subwords_by_pos.items():
    print(f"{pos}: {words}")
print()

#printing poetry
rules = {
    "origin": [
        "#perhaps_self# \n"
        "#perhaps_event# \n" 
        "#perhaps_absence# \n" 
        "#perhaps_beauty# \n" 
        "#perhaps_question# \n" 
    ],
    
    # "Perhaps it is simply that I am a coward."
    "perhaps_self": [
        "Perhaps it is simply that I am #noun.a#.",
        #"Perhaps it is #adv# that I am #noun.a#.",
        "Perhaps it is #adj# that I am #noun.a#."
        #"Perhaps it is #adv# that I am a coward."

    ],
    
    # "Perhaps it was a good thing."
    "perhaps_event": [
        "Perhaps it was #adj.a# thing.",
        "Perhaps it was #adj.a# #noun#.",
        "Perhaps it was #adj.a# #noun#, #adj#.",
    ],
    
    # "Perhaps in reality there was nothing for you to change into."
    "perhaps_absence": [
        "Perhaps in reality there was nothing for you to #verb# into.",
        "Perhaps in #noun# there was nothing for you to #verb# into.",
        "Perhaps in #noun# there was #noun# for you to #verb# into.",
        "Perhaps in #noun# there was nothing for you to change into.",
        "Perhaps in reality there was #noun# for you to change into.",

    ],
    
    # "Perhaps it is very beautiful, in fact."
    "perhaps_beauty": [
        "Perhaps it is very #adj#.", # didnt use in fact bc i dont like it
        "Perhaps #noun.a# is #adv# beautiful.",
        "Perhaps #noun.a# is #adv# #adj#.",
        #"Perhaps #noun.a# is #noun.a#",
    ],
    
    # "Perhaps that I am mad?"
    "perhaps_question": [
        "Perhaps that I am #adj#.", #period is more poetic
        "Perhaps that I am #noun.a#."
    ],
    
    "noun":      subwords_by_pos["NOUN"] if subwords_by_pos["NOUN"] else ["nothing"],
    "verb":      subwords_by_pos["VERB"] if subwords_by_pos["VERB"] else ["know", "suffer", "endure"],
   # "verb_3p":   [v + "s" for v in subwords_by_pos["VERB"]] if subwords_by_pos["VERB"] else ["knows", "suffers", "endures"],
   # "verb_past": [v + "ed" for v in subwords_by_pos["VERB"]] if subwords_by_pos["VERB"] else ["knew", "suffered", "endured"],
    "adj":       subwords_by_pos["ADJ"] if subwords_by_pos["ADJ"] else ["beautiful", "ugly", "empty", "spiteful", "alone"],
    "adv":       subwords_by_pos["ADV"] if subwords_by_pos["ADV"] else ["only", "simply", "really", "very", "always"],
}

grammar = tracery.Grammar(rules)
grammar.add_modifiers(base_english)
print()
print(grammar.flatten('#origin#'))

chosen word: understanding

 u n d e r s t a n d i n g
 u n d e r s t a n d      
           s t a n d i n g
 u n d e r                
     d   r     a     i n  
           s t a n d      
           s t a     i n  
           s t       i n g
     d e       a   d      
     d   r     a         g
         r     a     i n  
           s   a n d      
           s   a n       g
           s         i n g
   n           a         g
     d               i   g
       e         n d      
         r     a n        
         r     a         g
           s   a   d      
           s         i n  
               a n d      

----------- STAR -----------

 ~ * : ^ · . * + * · : o :
 ; ~ . . . · · ; * ·      
           + ^ * o ^ · ^ o
 ; x · * ·                
     ^   :     o     * ^  
           x + + ~ :      
           ^ · :     ; o  
           · ·       ~ ^ ;
     * ·       o   ;      
     o   .     .         ~
         x     o     : x  
           ;   o : o      
           ;   * ^     

## Trying Markov Chain generating poetry

I am not super happy with the tracery poem so I think i still want to try working a little bit more on markov chain generated poetry.
I actually tried this before my presentation in class but didn't end up with a good enough version of the poetry generator. But I want to try it again!

In [52]:
from shoestrings import TextGenerator
from shoestrings.tokenizers import SimpleCharacterTokenizer, SimpleWordTokenizer
print(word_choice)


understanding


In [53]:
# Generate Markov from those specific sentences

def my_boost(token, sequence, score):
    if token == 'was':
        return -100
    if token == 'were':
        return -100
    if token == 'I':
        return -100
    if token == 'am':
        return -100
    if token == 'that':
        return -100
    if token == 'a':
        return -100
    if token == 'it':
        return -100
    else:
        return score


gen2 = TextGenerator(n=2, tokenizer=SimpleWordTokenizer(), source=sentences)
gen3 = TextGenerator(n=3, tokenizer=SimpleWordTokenizer(), source=sentences)
gen4 = TextGenerator(n=4, tokenizer=SimpleWordTokenizer(), source=sentences)

gen7 = TextGenerator(n=7, tokenizer=SimpleWordTokenizer(), source=sentences)


poem = ""
phrase = ""
while not phrase.endswith("."):
    phrase = gen2.generate_one(start_string=word_choice, max_tokens=20)
poem = poem + phrase.capitalize()


phrase = ""
while not phrase.endswith("."):
    phrase = gen7.generate_one( max_tokens=30)
    #phrase = gen3.generate_one(start_string= "would you", max_tokens=30)


poem = poem + "\n" +phrase + "\n\nBut look, \n"

phrase = ""
while not phrase.endswith("."):
    verb_choice = random.choice(subwords_by_pos["VERB"] + ["know", "suffer", "endure","believe"])
    phrase = gen2.generate_one(start_string= verb_choice , max_tokens=10)
poem = poem + "I could not " + phrase 

phrase = ""
while not phrase.endswith(".") or len(phrase)<= 5:
    noun_choice = random.choice(subwords_by_pos["NOUN"] + ["nothing"])
    phrase = gen2.generate_one(start_string= noun_choice, custom_scorer=my_boost, max_tokens=30)
poem = poem + "\n" + "would you believe that " + phrase


phrase = ""
while not phrase.endswith(".") or len(phrase)<= 5:
    phrase = gen2.generate_one(start_string= "be", custom_scorer=my_boost, max_tokens=15)

poem = poem + "\n" + "would you " + phrase+ "\nwould you?"

#clean up
type(poem)
poem = poem.replace("- ", "-")
poem = poem.replace("' ", "'")
print(poem)



Understanding his foolishness we perverse out of her more independence of irony within me what it.
I knew a gentleman who prided himself all his life on being a connoisseur of Lafitte.

But look, 
I could not know.
would you believe that sing...
would you be out of respect for everything.
would you?
